In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import time
import matplotlib.pyplot as plt

from river import ensemble
from river import metrics
from river import preprocessing
from river import forest

In [ ]:
from pathlib import Path

dataset_name = "youtube_pedestrian.csv"
data_file = Path.cwd().parents[1] / "data" / "filtered" / dataset_name

df = pd.read_csv(data_file, parse_dates=["DATE"])
print("Loaded:", data_file)

In [ ]:
resolution_horizon_map = {
    "100ms": 96,
    "200ms": 48,
    "500ms": 20,
    "1000ms": 10,
    "2000ms": 5,
    "3000ms": 4,    
}


def resample_data(df, resolution):
    df_resampled = df.copy()

    df_resampled = df_resampled.set_index("DATE")
    df_resampled = df_resampled.resample(resolution).mean()

    df_resampled = df_resampled.dropna()

    df_resampled = df_resampled.round().astype(int)

    return df_resampled

In [ ]:
def run_arf(df, horizon, seed=42):
    split_point = int(len(df) * 0.8)
    test_start_time = df.index[split_point]

    model = preprocessing.StandardScaler() | forest.ARFRegressor(seed=seed)

    records = []

    feature_cols = [col for col in df.columns if col != "mac_dl_brate"]

    for t in range(horizon, len(df) - horizon):
        # Multi-horizon prediction
        for h in range(1, horizon + 1):
            idx = t + h
            row = df.iloc[idx]
            x = {col: row[col] for col in feature_cols}
            pred = model.predict_one(x)
            actual = row["mac_dl_brate"]
            timestamp = df.index[idx]
            source_t = df.index[t]

            records.append({
                "timestamp": timestamp,
                "source_t": source_t,
                "horizon": h,
                "actual": actual,
                "prediction": pred
            })

        # Training
            row = df.iloc[t + 1]
            x = {col: row[col] for col in feature_cols}
            y = row["mac_dl_brate"]
            model.learn_one(x, y)

    df_forecasts = pd.DataFrame(records)

    df_test_forecasts = df_forecasts[df_forecasts["source_t"] >= test_start_time].copy()

    df_test_forecasts = df_test_forecasts.dropna(subset=["actual", "prediction"])

    train_actuals = df_forecasts[df_forecasts["source_t"] < test_start_time]["actual"]
    scaler = MinMaxScaler()
    scaler.fit(train_actuals.values.reshape(-1, 1))

    actual_scaled = scaler.transform(df_test_forecasts["actual"].values.reshape(-1, 1)).flatten()
    pred_scaled = scaler.transform(df_test_forecasts["prediction"].values.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(actual_scaled, pred_scaled))
    mae = mean_absolute_error(actual_scaled, pred_scaled)

    return {
        "rmse": rmse,
        "mae": mae,
        "df_forecasts": df_forecasts,
        "df_test_forecasts": df_test_forecasts,
        "test_start_time": test_start_time,
        "split_point": split_point,
    }

In [ ]:
all_results = []

for resolution, horizon in resolution_horizon_map.items():
    print("=" * 60)
    print(f"Running ARF for resolution: {resolution}")
    print(f"Prediction horizon: {horizon}")

    df_resampled = resample_data(df, resolution)

    print("Resampled shape:", df_resampled.shape)

    if len(df_resampled) <= 2 * horizon:
        print(f"Skipping {resolution}: not enough rows for horizon {horizon}")
        continue

    result = run_arf(
        df=df_resampled,
        horizon=horizon,
        seed=42
    )

    rmse = result["rmse"]
    mae = result["mae"]

    print(f"Scaled RMSE: {rmse:.3f}")
    print(f"Scaled MAE : {mae:.3f}")

    all_results.append({
        "Model": "ARF",
        "Dataset": "youtube_pedestrian",
        "Resolution": resolution,
        "Prediction Horizon": horizon,
        "scaled_rmse": rmse,
        "scaled_mae": mae,
    })

In [ ]:
arf_temporal_results = pd.DataFrame(all_results)

arf_temporal_results

In [ ]:
results_dir = Path.cwd().parent / "results" / "metrics" / "temporal_resolution"
results_dir.mkdir(parents=True, exist_ok=True)

metrics_file = results_dir / "arf.csv"
arf_temporal_results.to_csv(metrics_file, index=False)

print("Saved metrics to:", metrics_file)